# parameter-subclass-of-tensor — worked example 2: filter trainable params by Parameter type

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `parameter-subclass-of-tensor`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`parameters()` and `buffers()` differ only by the registered type. Trainable params are exactly the attributes that are `isinstance(_, Parameter)`; plain MiniTensors (buffers, caches) and non-tensors are skipped. The strict `Parameter` filter is a subset of the broader `MiniTensor` filter.

## Worked solution

We reuse `MiniTensor` and `Parameter`, plus a tiny `Module` storing attributes in `__dict__`. `trainable_params(module)` walks the attributes and yields `(name, value)` only when `isinstance(value, Parameter)`. We build a module with a Parameter weight, a plain MiniTensor buffer, and an int config, then collect the trainables. Only the Parameter is yielded; the buffer and the int are skipped. We print the yielded names to confirm the strict filter selects exactly the trainable Parameter.

In [ ]:
import numpy as np

class MiniTensor:
    def __init__(self, array, requires_grad=False):
        self.array = np.asarray(array, dtype=float)
        self.requires_grad = requires_grad

class Parameter(MiniTensor):
    def __init__(self, array, requires_grad=True):
        super().__init__(array, requires_grad=requires_grad)

class Module:
    pass

def trainable_params(module):
    for name, val in module.__dict__.items():
        if isinstance(val, Parameter):
            yield name, val

m = Module()
m.weight = Parameter([1.0, 2.0])
m.running_mean = MiniTensor([0.0, 0.0])  # buffer, not trainable
m.num_layers = 3
print([name for name, _ in trainable_params(m)])